1.1 Extracting a Subset of the Dataset (30, 300)

In [3]:
import json
from collections import defaultdict

# Step 1: Extract restaurant businesses
restaurant_ids = set()
business_review_counts = defaultdict(set)

with open('../../yelp/business.json', 'r') as f:
    for line in f:
        biz = json.loads(line)
        if biz.get('categories') and 'Restaurants' in biz['categories']:
            restaurant_ids.add(biz['business_id'])

# Step 2: Count unique users per restaurant and filter restaurants with ≥300 users
user_reviews = defaultdict(set)
restaurant_user_counts = defaultdict(set)

with open('../../yelp/review.json', 'r') as f:
    for line in f:
        rev = json.loads(line)
        bid = rev['business_id']
        uid = rev['user_id']
        if bid in restaurant_ids:
            restaurant_user_counts[bid].add(uid)
            user_reviews[uid].add(bid)

# Final restaurant set
final_restaurants = {bid for bid, users in restaurant_user_counts.items() if len(users) >= 300}

# Step 3: Filter users who reviewed ≥30 of those restaurants
final_users = {uid for uid, bids in user_reviews.items() if len(bids & final_restaurants) >= 30}

# Step 4: Filter reviews between final users and final restaurants
filtered_reviews = []

with open('../../yelp/review.json', 'r') as f:
    for line in f:
        rev = json.loads(line)
        if rev['user_id'] in final_users and rev['business_id'] in final_restaurants:
            filtered_reviews.append(rev)

# Reporting counts
print("Users:", len(final_users))
print("Restaurants:", len(final_restaurants))
print("Reviews:", len(filtered_reviews))


Users: 3540
Restaurants: 3055
Reviews: 193887


1.2 Holdout

In [5]:
from datetime import datetime

# 1. Sort reviews by creation date (oldest first)
sorted_reviews = sorted(filtered_reviews, key=lambda x: datetime.strptime(x['date'], '%Y-%m-%d %H:%M:%S'))

# 2. Split: 20,000 oldest for training
train_reviews = sorted_reviews[:20000]

# 3. Remaining reviews
remaining = sorted_reviews[20000:]
mid = len(remaining) // 2

# 4. Split remaining into validation and test
val_reviews = remaining[:mid]
test_reviews = remaining[mid:]

# 5. Print sizes
print(f"Training: {len(train_reviews)}")
print(f"Validation: {len(val_reviews)}")
print(f"Test: {len(test_reviews)}")


Training: 20000
Validation: 86943
Test: 86944


1.1 Extracting a Subset of the Dataset (100, 1000)

2.1 Training Word2vec and FastText

In [6]:
# 1. Extract the texts
texts = [r['text'] for r in train_reviews]

# 2. Tokenize & lowercase
from nltk.tokenize import word_tokenize
tokenized = [word_tokenize(doc.lower()) for doc in texts]

# 3. Train the models
from gensim.models import Word2Vec, FastText
w2v, ft = Word2Vec(sentences=tokenized, seed=1234), FastText(sentences=tokenized, seed=1234)

# 4. Get similar words
for name, model in [('Word2Vec', w2v), ('FastText', ft)]:
    print(f"\n{name}")
    for target in ['tasty','give']:
        try:
            sims = model.wv.most_similar(target, topn=15)
            print(f" Top 15 near “{target}”: {[w for w,_ in sims]}")
        except KeyError:
            print(f" “{target}” not in vocab.")



Word2Vec
 Top 15 near “tasty”: ['yummy', 'delicious', 'delish', 'satisfying', 'good', 'flavorful', 'scrumptious', 'refreshing', 'boring', 'filling', 'strong', 'bland', 'decadent', 'divine', 'plentiful']
 Top 15 near “give”: ['consider', 'allow', 'add', 'lose', 'convince', 'giving', 'call', 'gave', 'ignore', 'make', 'bring', 'warn', 'suggest', 'feed', 'keep']

FastText
 Top 15 near “tasty”: ['tasty-', 'nasty', 'delicious', 'good-', 'unflavorful', 'yummy', 'good', 'flavorful', 'pasty', 'duhlicious', 'delicious-', 'goodbye', '-good', 'tasteful', 'yummy-']
 Top 15 near “give”: ['forgive', 'agave', 'given', 'gives', 'gave', 'gi', 'ive', 'giwa', 'ave', 'save', 'relive', 'remake', 'five', 'wave', 'nerve']


3.1 Implementation

In [7]:
import json
import random
import numpy as np

# 1. Reproducibility
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)

# 2. Preprocessing (as in §2.1)
from nltk.tokenize import word_tokenize

def preprocess_texts(reviews):
    """
    reviews: list of dicts, each with 'text' and 'stars'
    returns: list of token lists, and list of target ratings
    """
    texts = [r['text'] for r in reviews]
    ratings = [r['stars'] for r in reviews]
    tokenized = [word_tokenize(doc.lower()) for doc in texts]
    return tokenized, np.array(ratings)

# 3. Embedding training
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument

def train_embeddings(tokenized_texts,
                     method='word2vec',     # or 'fasttext' or 'doc2vec'
                     vec_size=100,
                     window=5,
                     epochs=5,
                     sg=1                   # 1 = skip-gram, 0 = CBOW (for word2vec/fasttext)
                     ):
    if method in ('word2vec', 'fasttext'):
        kwargs = {
            'vector_size': vec_size,
            'window':      window,
            'epochs':      epochs,
            'seed':        SEED,
            'sg':          sg
        }
        if method == 'word2vec':
            model = Word2Vec(sentences=tokenized_texts, **kwargs)
        else:
            model = FastText(sentences=tokenized_texts, **kwargs)
    else:  # doc2vec
        # tag each doc by its index
        tag_docs = [TaggedDocument(words=toks, tags=[i]) 
                    for i, toks in enumerate(tokenized_texts)]
        model = Doc2Vec(vector_size=vec_size,
                        window=window,
                        epochs=epochs,
                        seed=SEED,
                        dm=1,             # dm=1 for PV-DM (C-BOW like), 0 for PV-DBOW
                       )
        model.build_vocab(tag_docs)
        model.train(tag_docs,
                    total_examples=len(tag_docs),
                    epochs=epochs)
    return model

# 4. Document embedding aggregation
def embed_document(tokens, model, method='average'):
    """
    tokens: list of words (preprocessed)
    model: word2vec/fasttext OR doc2vec
    method: 'average', 'tfidf‐weighted', 'max_pool', or 'doc2vec'
    """
    if method == 'doc2vec':
        # tags in doc2vec are indices
        # assume you know the index for each document; here we just infer by retraining
        return model.infer_vector(tokens, epochs=model.epochs, alpha=model.alpha)
    
    # else we do word‐embeddings based
    vecs = []
    for w in tokens:
        if w in model.wv:
            vecs.append(model.wv[w])
        # else: skip OOV (you could also use a zero‐vector or random‐vector fallback)
    if not vecs:
        # no in‐vocab tokens → return zero‐vector
        return np.zeros(model.vector_size)
    
    arr = np.stack(vecs)
    if method == 'average':
        return arr.mean(axis=0)
    elif method == 'max_pool':
        return arr.max(axis=0)
    elif method == 'tfidf‐weighted':
        # you’d need a TF-IDF vectorizer fit on your corpus to supply weights per token
        raise NotImplementedError("Compute TF-IDF weights externally and apply here")
    else:
        raise ValueError(f"Unknown aggregation: {method}")

# 5. Full pipeline to matrix X, vector y
def build_feature_matrix(reviews,
                         embed_method='word2vec',
                         agg_method='average',
                         **embed_kwargs):
    toks, y = preprocess_texts(reviews)
    model = train_embeddings(toks, method=embed_method, **embed_kwargs)
    X = np.vstack([ embed_document(doc, model, method=agg_method) for doc in toks ])
    return X, y

# 6. (Optional) Feature manipulation
from sklearn.preprocessing import StandardScaler

def normalize_features(X_train, X_val=None, X_test=None):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    outs = [X_train_scaled]
    if X_val is not None:
        outs.append(scaler.transform(X_val))
    if X_test is not None:
        outs.append(scaler.transform(X_test))
    return outs if len(outs)>1 else X_train_scaled

# 7. Regression experiments
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split

# Split off a small hold-out from training as a local “validation” if you like:
# But assume we already have X_train, y_train, X_val, y_val, X_test, y_test.

# Example grid:
regressors = {
    'Ridge': {
       'model': Ridge(random_state=SEED),
       'params': {'alpha': [0.1, 1.0, 10.0]}
    },
    'RandomForest': {
       'model': RandomForestRegressor(random_state=SEED),
       'params': {'n_estimators': [50,100], 'max_depth':[None,10]}
    }
}

def evaluate_models(X_train, y_train, X_val, y_val):
    results = []
    for name, cfg in regressors.items():
        gs = GridSearchCV(cfg['model'], cfg['params'],
                          scoring='neg_mean_squared_error',
                          cv=3, n_jobs=-1)
        gs.fit(X_train, y_train)
        best = gs.best_estimator_
        val_mse = ((best.predict(X_val) - y_val)**2).mean()
        results.append((name, best, val_mse))
        print(f"{name}: best_params={gs.best_params_}, val_MSE={val_mse:.4f}")
    return results

# 8. Putting it all together
if __name__ == "__main__":
    # Assuming you’ve already read train_reviews, val_reviews, test_reviews
    X_train, y_train = build_feature_matrix(train_reviews,
                                           embed_method='fasttext',
                                           agg_method='average',
                                           vec_size=100,
                                           window=5,
                                           epochs=5,
                                           sg=1)
    X_val,   y_val   = build_feature_matrix(val_reviews,
                                           embed_method='fasttext',
                                           agg_method='average',
                                           vec_size=100,
                                           window=5,
                                           epochs=5,
                                           sg=1)
    X_test,  y_test  = build_feature_matrix(test_reviews,
                                           embed_method='fasttext',
                                           agg_method='average',
                                           vec_size=100,
                                           window=5,
                                           epochs=5,
                                           sg=1)

    # Normalize
    X_train, X_val, X_test = normalize_features(X_train, X_val, X_test)

    # Evaluate regressors
    results = evaluate_models(X_train, y_train, X_val, y_val)
    # Pick your best → final_test_mse:
    best_name, best_model, _ = min(results, key=lambda t: t[2])
    test_mse = ((best_model.predict(X_test) - y_test)**2).mean()
    print(f"\nFinal chosen: {best_name}, test MSE = {test_mse:.4f}")


Ridge: best_params={'alpha': 10.0}, val_MSE=3.5689
RandomForest: best_params={'max_depth': None, 'n_estimators': 100}, val_MSE=0.8745

Final chosen: RandomForest, test MSE = 1.2286


3.2 Hyperparameter tuning

In [9]:
import optuna
from sklearn.metrics import r2_score

# Define the Optuna objective function
def make_objective(embed_method, sg, dm=None):
    def objective(trial):
        # Sample hyperparameters
        vec_size = trial.suggest_int("vector_size", 50, 300)
        window   = trial.suggest_int("window", 2, 10)
        epochs   = trial.suggest_int("epochs", 5, 50)
        params = {
            "embed_method": embed_method,
            "vec_size": vec_size,
            "window": window,
            "epochs": epochs,
            "sg": sg
        }
        # FastText-specific
        if embed_method == "fasttext":
            min_n = trial.suggest_int("min_n", 3, 5)
            max_n = trial.suggest_int("max_n", 5, 8)
            params.update({"min_n": min_n, "max_n": max_n})
        # Doc2Vec-specific
        if embed_method.startswith("doc2vec"):
            params.pop("sg")
            params.update({"method": "doc2vec", "dm": dm})
        
        # Build features
        X_train, y_train = build_feature_matrix(train_reviews, **params)
        X_val,   y_val   = build_feature_matrix(val_reviews,   **params)

        # Normalize
        X_train, X_val = normalize_features(X_train, X_val)
        
        # Fit the chosen regressor (e.g., Ridge with alpha=1.0)
        model = Ridge(alpha=1.0, random_state=SEED)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        
        # Return validation R2
        return r2_score(y_val, preds)
    return objective

# Settings to tune
tuning_configs = [
    ("word2vec", 1, None, "word2vec-SGNS"),
    ("word2vec", 0, None, "word2vec-CBOW"),
    ("fasttext", 1, None, "fastText-SGNS"),
    ("fasttext", 0, None, "fastText-CBOW"),
    ("doc2vec", None, 1, "doc2vec-DM"),
    ("doc2vec", None, 0, "doc2vec-DBOW"),
]

results = {}

for method, sg, dm, study_name in tuning_configs:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(),
        study_name=study_name
    )
    study.optimize(make_objective(method, sg, dm), n_trials=50)
    # Record best params and R2
    results[study_name] = {
        "best_params": study.best_params,
        "best_r2": study.best_value
    }
    print(f"\n=== {study_name} ===")
    print("Best params:", study.best_params)
    print("Best R2   :", study.best_value)

# Optionally, access results for reporting
# results now contains the optimal hyperparameters and R2 for each setting.


[I 2025-06-26 08:11:52,676] A new study created in memory with name: word2vec-SGNS
[I 2025-06-26 08:28:54,183] Trial 0 finished with value: -19.33431144826697 and parameters: {'vector_size': 98, 'window': 7, 'epochs': 25}. Best is trial 0 with value: -19.33431144826697.
[I 2025-06-26 08:53:21,742] Trial 1 finished with value: -0.35834633084669276 and parameters: {'vector_size': 247, 'window': 9, 'epochs': 17}. Best is trial 1 with value: -0.35834633084669276.
[I 2025-06-26 09:39:36,276] Trial 2 finished with value: -46.06261466562408 and parameters: {'vector_size': 119, 'window': 9, 'epochs': 49}. Best is trial 1 with value: -0.35834633084669276.
[I 2025-06-26 09:59:35,213] Trial 3 finished with value: -0.17364649089489048 and parameters: {'vector_size': 269, 'window': 5, 'epochs': 28}. Best is trial 3 with value: -0.17364649089489048.
[I 2025-06-26 10:23:10,164] Trial 4 finished with value: -1.3604689347743077 and parameters: {'vector_size': 221, 'window': 8, 'epochs': 22}. Best is tr


=== word2vec-SGNS ===
Best params: {'vector_size': 269, 'window': 5, 'epochs': 28}
Best R2   : -0.17364649089489048


[I 2025-06-27 03:12:53,801] Trial 0 finished with value: -2.3213683562796286 and parameters: {'vector_size': 98, 'window': 7, 'epochs': 25}. Best is trial 0 with value: -2.3213683562796286.
[I 2025-06-27 03:18:22,085] Trial 1 finished with value: -1.2179390625915638 and parameters: {'vector_size': 247, 'window': 9, 'epochs': 17}. Best is trial 1 with value: -1.2179390625915638.
[I 2025-06-27 03:28:31,015] Trial 2 finished with value: -11.713965654985039 and parameters: {'vector_size': 119, 'window': 9, 'epochs': 49}. Best is trial 1 with value: -1.2179390625915638.
[I 2025-06-27 03:35:20,032] Trial 3 finished with value: -0.3533864206731372 and parameters: {'vector_size': 269, 'window': 5, 'epochs': 28}. Best is trial 3 with value: -0.3533864206731372.
[I 2025-06-27 03:41:42,084] Trial 4 finished with value: -1.687679592993761 and parameters: {'vector_size': 221, 'window': 8, 'epochs': 22}. Best is trial 3 with value: -0.3533864206731372.
[I 2025-06-27 03:44:35,488] Trial 5 finished wi


=== word2vec-CBOW ===
Best params: {'vector_size': 180, 'window': 6, 'epochs': 25}
Best R2   : -0.27122673596525515


[W 2025-06-27 08:47:23,697] Trial 0 failed with parameters: {'vector_size': 98, 'window': 7, 'epochs': 25, 'min_n': 5, 'max_n': 8} because of the following error: TypeError("train_embeddings() got an unexpected keyword argument 'min_n'").
Traceback (most recent call last):
  File "/Users/amirmohammad/.pyenv/versions/3.12.6/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/jh/k7f_w9415wb4g3myzy5dyxd80000gn/T/ipykernel_94688/2521847404.py", line 29, in objective
    X_train, y_train = build_feature_matrix(train_reviews, **params)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/jh/k7f_w9415wb4g3myzy5dyxd80000gn/T/ipykernel_94688/3769709048.py", line 101, in build_feature_matrix
    model = train_embeddings(toks, method=embed_method, **embed_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError:

TypeError: train_embeddings() got an unexpected keyword argument 'min_n'

Parallel

In [ ]:
import optuna
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge

# Define the Optuna objective function
def make_objective(embed_method, sg, dm=None):
    def objective(trial):
        # Sample hyperparameters
        vec_size = trial.suggest_int("vector_size", 50, 300)
        window   = trial.suggest_int("window", 2, 10)
        epochs   = trial.suggest_int("epochs", 5, 50)
        params = {
            "embed_method": embed_method,
            "vec_size": vec_size,
            "window": window,
            "epochs": epochs,
            "sg": sg,
            "workers": 4  # Use 4 threads for Word2Vec/FastText
        }

        # FastText-specific
        if embed_method == "fasttext":
            min_n = trial.suggest_int("min_n", 3, 5)
            max_n = trial.suggest_int("max_n", 5, 8)
            params.update({"min_n": min_n, "max_n": max_n})

        # Doc2Vec-specific
        if embed_method.startswith("doc2vec"):
            params.pop("sg")
            params.update({"method": "doc2vec", "dm": dm})
        
        # Build features
        X_train, y_train = build_feature_matrix(train_reviews, **params)
        X_val,   y_val   = build_feature_matrix(val_reviews,   **params)

        # Normalize
        X_train, X_val = normalize_features(X_train, X_val)
        
        # Fit the chosen regressor (e.g., Ridge with alpha=1.0)
        model = Ridge(alpha=1.0, random_state=SEED)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        
        # Return validation R2
        return r2_score(y_val, preds)
    return objective

# Settings to tune
tuning_configs = [
    ("word2vec", 1, None, "word2vec-SGNS"),
    ("word2vec", 0, None, "word2vec-CBOW"),
    ("fasttext", 1, None, "fastText-SGNS"),
    ("fasttext", 0, None, "fastText-CBOW"),
    ("doc2vec", None, 1, "doc2vec-DM"),
    ("doc2vec", None, 0, "doc2vec-DBOW"),
]

results = {}

for method, sg, dm, study_name in tuning_configs:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(),
        study_name=study_name
    )
    study.optimize(make_objective(method, sg, dm), n_trials=50, n_jobs=4)  # Parallel execution
    results[study_name] = {
        "best_params": study.best_params,
        "best_r2": study.best_value
    }
    print(f"\n=== {study_name} ===")
    print("Best params:", study.best_params)
    print("Best R2   :", study.best_value)


Evaluation

In [ ]:
import torch
import numpy as np
import pandas as pd
from torchmetrics.regression import (
    MeanAbsoluteError, MeanSquaredError, MeanAbsolutePercentageError,
    R2Score, ConcordanceCorrCoef, PearsonCorrCoef
)
from torchmetrics.retrieval import RetrievalNormalizedDCG
from scipy.stats import spearmanr, kendalltau

# Simulated predictions and targets (replace with actual predictions)
y_true = torch.tensor([4.0, 3.0, 5.0, 2.0, 1.0])
y_pred = torch.tensor([3.5, 3.2, 4.8, 2.2, 1.5])

# Regression metrics
metrics = {
    "MAE": MeanAbsoluteError(),
    "RMSE": MeanSquaredError(squared=False),
    "MAPE": MeanAbsolutePercentageError(),
    "R2": R2Score(),
    "CCC": ConcordanceCorrCoef(),
    "Pearson r": PearsonCorrCoef()
}
results = {name: metric(y_pred, y_true).item() for name, metric in metrics.items()}

# IR metrics (simulate multiple user queries)
user_ids = torch.tensor([0, 0, 1, 1, 1])  # group by users
for k in [1, 3, 5, 10]:
    ndcg = RetrievalNormalizedDCG(top_k=k)
    score = ndcg(y_pred, y_true.int(), indexes=user_ids).item()
    results[f"NDCG@{k}"] = score

# Rank correlation
results["Spearman ρ"], _ = spearmanr(y_true.numpy(), y_pred.numpy())
results["Kendall τ"], _ = kendalltau(y_true.numpy(), y_pred.numpy())

# Show results
df = pd.DataFrame([results])
print(df.T)
